# NSFW Adversarial Attack Demo (Image-based)

pNSFWMedia分類器に対する画像レベル敵対的攻撃のデモンストレーション。

## 準備
- 分類器: `models/target_classifier/pnsfwmedia_classifier.keras`
- 射影行列: `models/nudenet_projection.npy`
- NudeNet: `pip install nudenet`
- 攻撃対象画像: `dataset/nsfw_images/`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from src.pipeline import build_pipeline, IMAGE_SIZE
from src.utils import load_images, compute_metrics, set_seed, detect_gpu, setup_logging
from src.attacks.fgsm import FGSMAttack, FGSMConfig
from src.attacks.pgd import PGDAttack, PGDConfig
from src.attacks.cw import CWAttack, CWConfig
from src.attacks.deepfool import DeepFoolAttack, DeepFoolConfig

setup_logging('INFO')
set_seed(42)
detect_gpu()
print('TensorFlow:', tf.__version__)

## 1. パイプライン構築と画像読み込み

In [ ]:
# End-to-end pipeline (初回はONNX->TF変換が実行される)
pipeline = build_pipeline(
    projection_path='../models/nudenet_projection.npy',
    classifier_path='../models/target_classifier/pnsfwmedia_classifier.keras',
    backbone_cache_dir='../models/tf_backbone',
)

# 画像の読み込み
images, filenames = load_images('../dataset/nsfw_images', max_images=50)
print(f'Images shape: {images.shape}')

# 元の予測
original_probs = pipeline.predict_images(images)
nsfw_mask = original_probs >= 0.5
print(f'NSFW samples: {nsfw_mask.sum()}/{len(original_probs)}')
print(f'Mean NSFW probability: {original_probs[nsfw_mask].mean():.4f}')

## 2. 元の予測分布

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(original_probs, bins=30, alpha=0.7, edgecolor='black')
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold (0.5)')
ax.set_xlabel('NSFW Probability', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Original NSFW Probability Distribution', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 3. FGSM攻撃

In [ ]:
epsilons_px = [4, 8, 16, 32]  # in /255
fgsm_results = {}

for eps_px in epsilons_px:
    eps = eps_px / 255.0
    config = FGSMConfig(epsilon=eps, batch_size=8)
    attack = FGSMAttack(pipeline, config)
    adv, noise = attack.attack(images)
    adv_probs = pipeline.predict_images(adv)
    metrics = compute_metrics(original_probs, adv_probs, noise)
    fgsm_results[eps_px] = {'metrics': metrics, 'adv_probs': adv_probs, 'adv': adv}
    sr = metrics.get('success_rate_0.5', 0)
    print(f'FGSM eps={eps_px}/255: SR@0.5={sr:.2%}')

In [ ]:
# Visualise: original vs adversarial
best_eps = 8
adv_images = fgsm_results[best_eps]['adv']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(images[i])
    axes[0, i].set_title(f'Original\np={original_probs[i]:.3f}', fontsize=10)
    axes[0, i].axis('off')
    adv_p = pipeline.predict_images(adv_images[i:i+1])[0]
    axes[1, i].imshow(adv_images[i])
    axes[1, i].set_title(f'FGSM (eps={best_eps}/255)\np={adv_p:.3f}', fontsize=10)
    axes[1, i].axis('off')
plt.suptitle('Original vs Adversarial Images', fontsize=14)
plt.tight_layout()
plt.show()

## 4. PGD攻撃

In [ ]:
pgd_results = {}
for eps_px in epsilons_px:
    eps = eps_px / 255.0
    config = PGDConfig(epsilon=eps, alpha=eps/4, iterations=20, batch_size=8)
    attack = PGDAttack(pipeline, config)
    adv, noise, iters = attack.attack(images)
    adv_probs = pipeline.predict_images(adv)
    metrics = compute_metrics(original_probs, adv_probs, noise)
    metrics['avg_iterations'] = float(iters.mean())
    pgd_results[eps_px] = {'metrics': metrics, 'adv_probs': adv_probs}
    sr = metrics.get('success_rate_0.5', 0)
    print(f'PGD eps={eps_px}/255: SR@0.5={sr:.2%}, avg_iters={iters.mean():.1f}')

In [ ]:
# FGSM vs PGD comparison
fig, ax = plt.subplots(figsize=(10, 5))
for name, results in [('FGSM', fgsm_results), ('PGD', pgd_results)]:
    rates = [results[e]['metrics'].get('success_rate_0.5', 0) for e in epsilons_px]
    ax.plot(epsilons_px, rates, 'o-', label=name, linewidth=2)
ax.set_xlabel('Epsilon (pixel /255)', fontsize=12)
ax.set_ylabel('Success Rate @ 0.5', fontsize=12)
ax.set_title('FGSM vs PGD: Attack Success Rate', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

## 5. C&W攻撃 (サブセット)

In [ ]:
demo_images = images[:10]
demo_probs = original_probs[:10]

config = CWConfig(c=1.0, kappa=0.0, iterations=500, batch_size=2)
attack = CWAttack(pipeline, config)
adv, noise, iters = attack.attack(demo_images)
adv_probs = pipeline.predict_images(adv)
metrics = compute_metrics(demo_probs, adv_probs, noise)
print(f'C&W: SR@0.5={metrics.get("success_rate_0.5", 0):.2%}')
print(f'Avg L2 norm: {metrics.get("avg_l2_norm", 0):.4f}')

## 6. DeepFool攻撃 (サブセット)

In [ ]:
config = DeepFoolConfig(max_iterations=100, overshoot=0.02)
attack = DeepFoolAttack(pipeline, config)
adv, noise, iters = attack.attack(demo_images)
adv_probs = pipeline.predict_images(adv)
metrics = compute_metrics(demo_probs, adv_probs, noise)
print(f'DeepFool: SR@0.5={metrics.get("success_rate_0.5", 0):.2%}')
print(f'Avg L2 norm: {metrics.get("avg_l2_norm", 0):.4f}')
print(f'Avg iterations: {iters.mean():.1f}')

## 7. 比較サマリー

In [ ]:
print(f'{"Method":<12} {"SR@0.5":>8} {"SR@0.3":>8} {"Avg L-inf(px)":>14}')
print('-' * 46)
for name, results in [('FGSM', fgsm_results), ('PGD', pgd_results)]:
    for eps_px in epsilons_px:
        m = results[eps_px]['metrics']
        print(f'{name} {eps_px}/255   '
              f'{m.get("success_rate_0.5", 0):>8.2%} '
              f'{m.get("success_rate_0.3", 0):>8.2%} '
              f'{m.get("avg_linf_norm_pixel", 0):>14.2f}')